In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu, false_discovery_control

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "dystroglycan_violin"
OUT_DIR.mkdir(parents=True, exist_ok=True)

fs.DPI = 500
plt.rcParams.update({
    "figure.dpi": fs.DPI,
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6, "figure.titlesize": 6, "legend.title_fontsize": 6,
    "xtick.labelsize": 5.5, "ytick.labelsize": 5.5, "legend.fontsize": 5.5,
})

ORDER = ["SF", "C-ion", "X-ray"]
COLORS = {"SF": fs.LIGHTGREY, "C-ion": fs.CION, "X-ray": fs.XRAY}
PANEL_MM = (75, 80)

In [ ]:
df = pd.read_csv(BASE / "dystroglycan_vascular_area_fraction.csv")
values = {g: df.loc[df["group"] == g, "area_fraction_pct"].to_numpy() for g in ORDER}

summary = df.groupby("group")["area_fraction_pct"].describe().loc[ORDER, ["count", "50%", "25%", "75%"]]
summary.columns = ["n", "median", "Q1", "Q3"]
summary = summary.astype({"n": int}).round(2)
summary.to_csv(OUT_DIR / "summary.csv")
summary

In [ ]:
h, p_kw = kruskal(*values.values())
print(f"Kruskal-Wallis: H = {h:.2f}, p = {p_kw:.3g}")

pairs = [("SF", "C-ion"), ("SF", "X-ray"), ("C-ion", "X-ray")]
tests = [mannwhitneyu(values[a], values[b], alternative="two-sided") for a, b in pairs]
pairwise = pd.DataFrame({
    "group1": [a for a, _ in pairs],
    "group2": [b for _, b in pairs],
    "U": [t.statistic for t in tests],
    "p": [t.pvalue for t in tests],
})
pairwise["p_BH"] = false_discovery_control(pairwise["p"], method="bh")
pairwise.to_csv(OUT_DIR / "pairwise_mannwhitney.csv", index=False)
pairwise

In [ ]:
def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

fig, ax = plt.subplots(figsize=(PANEL_MM[0] * fs.MM, PANEL_MM[1] * fs.MM), layout="constrained")
sns.violinplot(data=df, x="group", y="area_fraction_pct", order=ORDER, hue="group", palette=COLORS,
               legend=False, cut=0, inner=None, linewidth=0.5, saturation=1, ax=ax)
np.random.seed(0)
sns.stripplot(data=df, x="group", y="area_fraction_pct", order=ORDER, color="black",
              size=1.5, jitter=0.15, ax=ax)

x = {g: i for i, g in enumerate(ORDER)}
levels = {("SF", "C-ion"): 0, ("C-ion", "X-ray"): 0, ("SF", "X-ray"): 1}
for _, row in pairwise.iterrows():
    y = 105 + 9 * levels[(row["group1"], row["group2"])]
    x1, x2 = x[row["group1"]] + 0.04, x[row["group2"]] - 0.04
    ax.plot([x1, x1, x2, x2], [y, y + 2, y + 2, y], color="black", lw=0.5)
    ax.text((x1 + x2) / 2, y + 2, stars(row["p_BH"]), ha="center", va="bottom", fontsize=7)

ax.set_xlabel("")
ax.tick_params(axis="x", labelsize=7)
ax.set_ylabel("β-dystroglycan-positive area (%)")
ax.set_yticks(range(0, 101, 20))
ax.spines["left"].set_bounds(0, 100)
ax.set_ylim(-2, 105 + 9 + 10)
with plt.rc_context({"savefig.bbox": "standard", "savefig.pad_inches": 0}):
    fs.save(fig, OUT_DIR / "violin_vascular")